# Grip 数据接口验证

这个笔记本验证 `grip_data_interface.py` 是否能够按团队约定加载 R09/R11，并演示后续分析代码如何读取变长 trial EEG、连续握力、元数据和事件。

## 1. Setup

验证使用前 20 个通道控制内存占用；接口仍会报告原始通道总数。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from grip_data_interface import load_flight_trials, select_flight_split

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "0807华山grip flight"
VALIDATION_CHANNELS = np.arange(20)

RUN_FILES = {
    "R09": ("testS001R09.dat", "testS001R09_1.dat"),
    "R11": ("testS001R11.dat", "testS001R11_1.dat"),
}

for run_id, filenames in RUN_FILES.items():
    for filename in filenames:
        path = DATA_DIR / filename
        assert path.exists(), f"Missing {run_id} input: {path}"

print("project:", PROJECT_ROOT)
print("data:", DATA_DIR)
print("validation channels:", VALIDATION_CHANNELS.tolist())

## 2. Load R09 and R11

`load_flight_trials()` 返回与 playgroundgit `pen_split` 相同风格的变长 trial 字典。

In [ ]:
runs = {}
for run_id, (eeg_name, task_name) in RUN_FILES.items():
    runs[run_id] = load_flight_trials(
        DATA_DIR / eeg_name,
        DATA_DIR / task_name,
        split_name="all",
        segment="flight",
        channel_indices=VALIDATION_CHANNELS,
    )
    print(
        run_id,
        f"interface={runs[run_id]['interface_version']}",
        f"trials={len(runs[run_id]['X_list'])}",
        f"channels={runs[run_id]['n_ch']}/{runs[run_id]['raw_n_ch']}",
        f"sr={runs[run_id]['sr']:g} Hz",
    )

## 3. Inspect the interface

最常用字段是 `X_list`、`target_list`、`meta`、`events`、`sr` 和 `time_list`。

In [ ]:
interface_rows = []
for run_id, data in runs.items():
    interface_rows.append({
        "run": run_id,
        "interface_version": data["interface_version"],
        "n_trials": len(data["X_list"]),
        "loaded_channels": data["n_ch"],
        "raw_channels": data["raw_n_ch"],
        "sampling_rate_hz": data["sr"],
        "target_names": data["target_names"],
        "state_names": data["state_names"],
    })

interface_summary = pd.DataFrame(interface_rows)
display(interface_summary)

r11 = runs["R11"]
print("R11 first EEG shape:", r11["X_list"][0].shape)
print("R11 first target shape:", r11["target_list"][0].shape)
print("R11 first state fields:", list(r11["state_list"][0]))

## 4. Validate trial-level invariants

每个 trial 的 EEG、握力目标、归一化握力和时间轴必须逐点等长。

In [ ]:
check_rows = []
for run_id, data in runs.items():
    assert data["interface_version"] == "1.0"
    assert "code" not in data["events"].columns
    assert len(data["meta"]) == len(data["X_list"])

    for local_i, (eeg, target, normalized, time_s, states) in enumerate(zip(
        data["X_list"],
        data["target_list"],
        data["force_normalized_list"],
        data["time_list"],
        data["state_list"],
    )):
        n_samples = eeg.shape[1]
        assert target.shape == (1, n_samples)
        assert normalized.shape == (n_samples,)
        assert time_s.shape == (n_samples,)
        assert all(values.shape == (n_samples,) for values in states.values())
        assert np.isfinite(target).all()
        row = data["meta"].iloc[local_i]
        assert int(row.n_samples) == n_samples
        check_rows.append({
            "run": run_id,
            "trial_id": int(row.trial_id),
            "outcome": row.outcome,
            "collision": bool(row.collision),
            "eeg_shape": str(eeg.shape),
            "target_shape": str(target.shape),
            "duration_s": float(row.duration_s),
        })

trial_checks = pd.DataFrame(check_rows)
display(trial_checks)
print("PASS: all trial-level arrays are aligned and equal length.")

## 5. Validate outcome splits and events

In [ ]:
split_rows = []
event_tables = []
for run_id, data in runs.items():
    counts = {name: len(select_flight_split(data, name)["X_list"]) for name in ("all", "success", "failure", "collision")}
    split_rows.append({"run": run_id, **counts})
    events = data["events"].copy()
    events.insert(0, "run", run_id)
    event_tables.append(events)

split_summary = pd.DataFrame(split_rows)
events_summary = pd.concat(event_tables, ignore_index=True)
display(split_summary)
display(events_summary[["run", "trial_id", "event", "label", "alignment_error_ms"]])

expected = {"R09": {"all": 3, "success": 2, "failure": 1, "collision": 1}, "R11": {"all": 6, "success": 1, "failure": 5, "collision": 5}}
for row in split_summary.to_dict(orient="records"):
    run_id = row.pop("run")
    assert row == expected[run_id]
assert events_summary["alignment_error_ms"].abs().max() <= 0.5
print(f"PASS: split counts match; max event alignment error = {events_summary['alignment_error_ms'].abs().max():.4f} ms.")

## 6. Minimal downstream usage

下面示范从接口取出一个 trial。后续特征提取应以整个 trial 为训练/测试划分单位。

In [ ]:
trial_index = 1
eeg_trial = r11["X_list"][trial_index]
force_trial = r11["target_list"][trial_index][0]
time_s = r11["time_list"][trial_index]
trial_meta = r11["meta"].iloc[trial_index]
trial_events = r11["event_list"][trial_index]

print(trial_meta[["trial_key", "outcome", "collision", "duration_s"]])
print("EEG:", eeg_trial.shape, "Force:", force_trial.shape)
display(trial_events[["event", "label", "eeg_time_s", "alignment_error_ms"]])

In [ ]:
show_s = min(10.0, float(time_s[-1]))
show = time_s <= show_s
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, constrained_layout=True)

for channel_i in range(min(3, eeg_trial.shape[0])):
    signal = eeg_trial[channel_i, show].astype(float)
    scale = np.nanstd(signal) or 1.0
    axes[0].plot(time_s[show], (signal - np.nanmean(signal)) / scale + channel_i * 5, lw=0.6, label=r11["channel_names"][channel_i])
axes[0].set_ylabel("EEG z-score + offset")
axes[0].legend(loc="upper right", ncol=3)
axes[0].set_title(f"{trial_meta.trial_key}: interface output preview")

axes[1].plot(time_s[show], force_trial[show], color="tab:orange", lw=1.2)
axes[1].set_ylabel("Grip force")
axes[1].set_xlabel("Time from flight start (s)")
axes[1].set_xlim(0, show_s)
plt.show()

## 7. Result

验证通过后，团队分析代码只需依赖 `load_flight_trials()` 返回的字典，不需要重复实现 DAT 解析、SourceTime 对齐、trial 切分或事件标签转换。